# 1.消息类型

在LangChain中，发送给LLM的消息、LLM返回的消息都统一被封装为BaseMessage，它是Agent中基本的上下文单元。

在LangChain中，我们并不需要自己创建BaseMessage对象，LangChain已经把常见消息根据角色（Role）创建了对应的BaseMessage的子类：
- SystemMessage：role是system，代表系统消息，用于设定模型角色和交互背景
- HumanMessage：role是user，代表用户输入的消息
- AIMessage：role是assistant，代表LLM生成的响应，包含：文本、工具调用、元数据
- ToolMessage：role是tool，代表工具调用时产生的结果

我们可以直接使用这些Messages类型来发送消息。

In [11]:
from tkinter.constants import MULTIPLE

from IPython.terminal.shortcuts.auto_suggest import accept
from dashscope.common.env import api_key
from dotenv import load_dotenv
import os

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain.agents import create_agent
from langchain_core.messages import SystemMessage, HumanMessage
from openai import base_url

# 1. 加载环境变量
load_dotenv()       # 加载环境变量
qwen_api_key = os.getenv("ALIYUN")


# 2、定义工具
def get_weather(location:str) -> str:
    """
    获取指定城市的天气
    :param location:
    :return:
    """
    return f"当前{location}天气晴朗"

# 3. 创建agent
qwen_model = init_chat_model(model='qwen3.8-max-0902', model_provider="openai",
                        base_url="https://maas.qianwenaiapi.com/compatible-mode/v1", api_key=qwen_api_key, )

agent = create_agent(model=qwen_model, tools=[get_weather], )

# 4.发起调用
response = agent.invoke({"messages": [SystemMessage("请使用工具来获取天气信息"), HumanMessage("请问深圳天气怎么样？")]})


In [24]:
# print(response)
# print(response['messages'])
# print(len(response['messages']))
for message in response['messages']:
    message.pretty_print()

================================ System Message ================================

请使用工具来获取天气信息
================================ Human Message =================================

请问深圳天气怎么样？
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_1d2d9591d53249a0a29038eb)
 Call ID: call_1d2d9591d53249a0a29038eb
  Args:
    location: 深圳
================================= Tool Message =================================
Name: get_weather

当前深圳天气晴朗
================================== Ai Message ==================================

深圳当前天气：**晴朗** ☀️

天气状况良好，适合外出活动。需要我查询其他城市的天气吗？


# 2.多模态消息

之前我们都是向模型发送文本消息，但是 LangChain 也支持向模型发送多模态消息，比如图片、音频、视频、文本等。但前提是必须是多模态模型才支持。

一些支持多模态的模型有：
- qwen3.5-plus
- gpt-5-nano
- ...

我们以qwen3.5-plus为例，演示向模型发送图片消息

## 2.1.在线图片
首先，我们演示如何发送一个在线图片给模型，也就是指定模型的url地址。
图片如下：

<img src="https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg" width="500" height="300" alt="图片描述">



In [25]:
# 创建模型
model = init_chat_model(model='qwen3.5-plus',
                        model_provider="openai",
                        base_url="https://maas.qianwenaiapi.com/compatible-mode/v1", api_key=qwen_api_key, )
# 创建agent
agent = create_agent(model=model, )


In [ ]:
# 准备多模态消息
message = HumanMessage([
        {"type": "text", "text": "描述一下这张图片的内容."},
        {"type": "image", "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"},  # 这是在线图片
    ])


In [26]:
message = HumanMessage([{"type": "text", "text": "描述一下这张图片的内容."}, {"type": "image",
                                                                                    "url": "https://help-static-aliyun-doc.aliyuncs.com/file-manage-files/zh-CN/20241022/emyrja/dog_and_girl.jpeg"}, ])

In [31]:
stream = agent.stream(
    {'messages':[message]},
    stream_mode='messages',
)
for chunk,metadata in stream:
    if chunk.content:
        print(chunk.content,end='',flush=True)

这张图片展现了一幅温馨的海滩人宠互动画面：

**主体内容：**
- 一位长发年轻女子坐在沙滩上，身穿黑白格子衬衫和深色长裤，赤着脚，面带灿烂的笑容。
- 她面前坐着一只浅黄色（奶油色）的拉布拉多犬，狗狗抬起一只前爪，与女子伸出的手掌相触，像是在"击掌"或"握手"，互动十分亲密有趣。
- 狗狗身上佩戴着带花纹的胸背带（牵引背带），一条红色牵引绳散落在旁边的沙地上。
- 女子手腕上戴着白色手表，另一只手似乎拿着零食或玩具，可能正在训练或奖励狗狗。

**环境与光线：**
- 背景是开阔的海面，可以看到一道海浪正在翻卷拍打岸边。
- 天空明亮泛白，画面右侧有温暖的金色阳光洒入（似日落或日出时分），形成柔和的逆光效果，为女子的头发和轮廓镀上一层暖光。
- 沙滩上布满脚印和起伏的痕迹，显得自然松弛。

**整体氛围：**
画面色调柔和温暖，构图简洁，传递出人与宠物之间信任、陪伴与快乐的情感，是一张充满温情的生活化摄影作品。

## 2.2.本地图片数据(没有成功）
有时候用户会上传图片数据，而不是图片的url地址。我们需要将图片数据转换成base64字符串，然后发送给模型。

接下来我们会模拟图片上传、转换的过程。

首先，我们安装一个上传组件，用于模拟图片上传。

In [5]:
from ipywidgets import FileUpload
from IPython.display import display

uploader = FileUpload(accept='*',multiple=False)
display(uploader)


FileUpload(value=(), accept='*', description='Upload')

In [7]:
print(uploader.value)

()


In [11]:
# 读取图片，转为base64字符串
import base64

# 获取第一个（也是唯一一个）上传的文件
uploaded_file = uploader.value[0]

# 获取其内存视图
content_mv = uploaded_file["content"]

# 转换内存视图->字节
img_bytes = bytes(content_mv)  # or content_mv.tobytes()

# base64编码
img_b64 = base64.b64encode(img_bytes).decode("utf-8")

❌ 还没有上传文件！请在上一个单元格选择图片。
